# 01 — Verify Annotations
Sanity check: parse the CVAT XML and visualize landmarks on a sample image.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import numpy as np

from src.phase1.cvat_parser import parse_cvat_xml, KEYPOINT_NAMES
from src.utils.io import load_config

cfg = load_config('../config.yaml')
records = parse_cvat_xml(cfg['data']['annotation_file'])
print(f'Parsed {len(records)} images')

In [ ]:
# Summary table
import pandas as pd

rows = []
for r in records:
    rows.append({
        'image_id': r['image_id'],
        'patient_id': r['patient_id'],
        'timepoint': r['timepoint'],
        'has_calibration': r['has_calibration'],
        'has_landmarks': r['has_landmarks'],
        'n_valid_kp': sum(r['valid_mask']),
        'treatment': ', '.join(r['treatment']) if r['treatment'] else '',
        'quality_flags': ', '.join(r['quality_flags']) if r['quality_flags'] else '',
    })

df = pd.DataFrame(rows)
print(df.describe())
df.head(20)

In [ ]:
# Visualize landmarks on first annotated image
annotated = [r for r in records if r['has_landmarks']]
if not annotated:
    print('No annotated images yet — Dr. is still annotating.')
else:
    rec = annotated[0]
    img_path = os.path.join('..', cfg['data']['image_dir'], rec['file_name'])
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(figsize=(10, 12))
    ax.imshow(img, cmap='gray')

    colors = plt.cm.tab10.colors
    patches = []
    for i, kp in enumerate(rec['keypoints']):
        if kp['visible']:
            ax.scatter(kp['x'], kp['y'], c=[colors[i % 10]], s=80, zorder=5)
            ax.annotate(kp['name'], (kp['x'], kp['y']), fontsize=8, color=colors[i % 10],
                        xytext=(5, 5), textcoords='offset points')
            patches.append(mpatches.Patch(color=colors[i % 10], label=kp['name']))

    if rec['calibration_points']:
        p1, p2 = rec['calibration_points']
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'y-', linewidth=2, label='Calibration_30mm')

    ax.legend(handles=patches, loc='upper left', fontsize=7)
    ax.set_title(f"{rec['image_id']} — {rec['treatment']}")
    plt.tight_layout()
    plt.show()